# Severe Rainfall Frequency Analysis (PRISM)

This notebook identifies extreme rainfall events in observed PRISM precipitation data and quantifies **annual event frequency**.

Extreme events are defined using a **fixed daily precipitation threshold**, applied consistently across time to support non-stationary analysis.

This notebook produces **event counts only**.  
Comparison across historical periods (“epochs”) is performed in `03_epoch_comparison.ipynb`.

---

## Purpose

- Flag extreme rainfall events using a fixed threshold
- Count extreme events by calendar year
- Generate frequency tables for downstream comparison and risk analysis

---

## Key Concepts

- **Frequency vs. intensity**  
  This analysis evaluates how often extreme events occur, not storm magnitude.

- **Consistency across time**  
  A fixed threshold is used to detect change without assuming stationarity.

- **Modularity**  
  Outputs are exported for use in later notebooks and external replication.

---

## Inputs

- Cleaned daily precipitation table from:
  - `01_prism_precip_ingest.ipynb`

---

## Outputs

- Annual extreme rainfall event counts
- Tables suitable for:
  - epoch comparison
  - mapping
  - replication in other regions

---

## Notes

- This notebook does not perform epoch-based aggregation.
- Comparison across historical periods is performed in `03_epoch_comparison.ipynb`.


In [1]:
from pathlib import Path
import pandas as pd
import sys
import os

# Resolve repository root (notebooks/ is one level down)
REPO_ROOT = Path("..").resolve()

# Ensure src/ is importable
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Core repo directories
DATA_DIR = REPO_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
DATA_SAMPLE_DIR = DATA_DIR / "sample"

OUTPUTS_DIR = REPO_ROOT / "outputs"
TABLES_DIR = OUTPUTS_DIR / "tables"
MAPS_DIR = OUTPUTS_DIR / "maps"

DOCS_DIR = REPO_ROOT / "docs"
SAMPLE_DIR = REPO_ROOT / "sample_data"

# Create expected output folders (safe if they already exist)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
MAPS_DIR.mkdir(parents=True, exist_ok=True)



from src.prism_extremes import (
    flag_extreme_events,
    count_annual_extremes
)

print("Repo root on path:", REPO_ROOT)
print("Notebook running successfully.")
print("Repo root:", REPO_ROOT)
print("Data dir:", DATA_DIR)
print("Outputs dir:", OUTPUTS_DIR)

Repo root on path: C:\Users\admin\Documents\GitHub\nonstationary-flood-risk-framework
Notebook running successfully.
Repo root: C:\Users\admin\Documents\GitHub\nonstationary-flood-risk-framework
Data dir: C:\Users\admin\Documents\GitHub\nonstationary-flood-risk-framework\data
Outputs dir: C:\Users\admin\Documents\GitHub\nonstationary-flood-risk-framework\outputs


In [2]:
# -----------------------------
# USER CONFIGURATION
# -----------------------------

# Input from Notebook 01
CLEANED_PRISM_PATH = OUTPUTS_DIR / "tables" / "prism_daily_cleaned.csv"

# Column names (must match Notebook 01 output)
DATE_COL = "Date"
PRECIP_COL = "inchesPpt"

# Extreme rainfall threshold (inches)
EXTREME_THRESHOLD_IN = 3.0

In [3]:
if not CLEANED_PRISM_PATH.exists():
    raise FileNotFoundError(
        f"Missing cleaned PRISM data at {CLEANED_PRISM_PATH}. "
        "Run 01_prism_precip_ingest.ipynb first."
    )

In [4]:
df = pd.read_csv(
    CLEANED_PRISM_PATH,
    parse_dates=[DATE_COL]
).set_index(DATE_COL)

df.head()

,inchesPpt
Date,
1981-01-01,0.0
1981-01-02,0.0
1981-01-03,0.0
1981-01-04,0.0
1981-01-05,0.0


In [5]:
# Flag extreme-event days
extreme_flags = flag_extreme_events(
    df,
    precip_col=PRECIP_COL,
    threshold_in=EXTREME_THRESHOLD_IN
)

print("Total extreme-event days:", int(extreme_flags.sum()))

Total extreme-event days: 12


In [6]:
# Count extreme events by calendar year
annual_counts = count_annual_extremes(extreme_flags)

annual_counts.head()


C:\Users\admin\Documents\GitHub\nonstationary-flood-risk-framework\src\prism_extremes.py:98: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  return extreme_flags.resample("Y").sum().astype(int)


Date
1981-12-31    0
1982-12-31    0
1983-12-31    0
1984-12-31    0
1985-12-31    0
Freq: YE-DEC, Name: inchesPpt, dtype: int64

In [7]:
annual_counts_df = annual_counts.reset_index()
annual_counts_df.columns = ["year", "extreme_event_count"]
annual_counts_df["year"] = annual_counts_df["year"].dt.year

annual_counts_df.head()

,year,extreme_event_count
0,1981,0
1,1982,0
2,1983,0
3,1984,0
4,1985,0


In [8]:
OUTPUT_PATH = OUTPUTS_DIR / "tables" / "prism_annual_extreme_counts.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

annual_counts_df.to_csv(OUTPUT_PATH, index=False)

print(f"Annual extreme-event counts saved to {OUTPUT_PATH}")

Annual extreme-event counts saved to C:\Users\admin\Documents\GitHub\nonstationary-flood-risk-framework\outputs\tables\prism_annual_extreme_counts.csv


In [9]:
annual_counts_df.describe()

,year,extreme_event_count
count,41.000000,41.000000
mean,2001.000000,0.292683
std,11.979149,0.558744
min,1981.000000,0.000000
25%,1991.000000,0.000000
50%,2001.000000,0.000000
75%,2011.000000,0.000000
max,2021.000000,2.000000


## Next Steps

This notebook produces annual extreme rainfall event counts derived from observed PRISM data using a fixed threshold definition.

These annual frequency outputs are used directly in:

- `03_epoch_comparison.ipynb`  
  → to compare extreme-event frequency across historical time windows without assuming stationarity

Users may proceed directly to the next notebook after confirming this step completes successfully.